In [ ]:
# ---- which sweeps "Run All" runs: True = run, False = skip -------------------------------------
# every sweep cell below checks its flag; a staged sweep runs all its stages in order, each on top of the best values
# of the stages before
sweep_lightgbm_gap_model = True                 # 9 + 3 + 3 runs in 3 stages (CPU)

# Task 6 hyperparameter sweeps

**Run All:** set the flags in the first cell, then run the whole notebook. Every sweep cell only runs if its flag is `True`.
A staged sweep runs all its stages in order; every stage passes the best values of the stages before on as constants
(`best_values_of`), so it does not depend on copying values into `lib/config.py` in between. Run a single staged cell by hand
only after the stages before it (it needs their `..._best_values`). A sweep that fails stops Run All there; its finished runs are
in its output file.

One cell per model. `run_grid_search(task, model, parameter_grid)`:

- **a list** in `parameter_grid` = values to try (every combination of all lists is run)
- **a single value** (or a one-element list) = constant in every run
- **not mentioned** = the value from `submission/lib/config.py` (Task 6 section)
- **nested settings** as `"outer.inner"`, e.g. `"temporal_biases.drift_regularization"`

Every run trains on the training split and is scored on the validation split: **MAE in days on every user's last gap (lower is better)**.
No refit on all data, nothing is written to `trained_models/` -> the submitted models stay untouched.

**Result:** `output/task6/<model>_<start time>.json` with the best combination and its full parameter set
(`best.parameters`, can be copied into `lib/config.py`), the swept and constant values and the score of every run.
For a staged sweep: copy `best.parameters` of its last stage (before a noise check) - it contains the earlier stages' best values.
The file is rewritten after every run: interrupting a cell keeps the finished runs.

**Optional:** there is no finalist check: Task 6 has one fixed validation split (every user's last gap). Sweeping `seed` (the fold assignment of the out-of-fold score) shows how much the score moves by chance.
`show_training_log=True` prints the training's own log lines of every run.

The score is the model's own out-of-fold MAE on the validation rows (as in `task6_ensemble.ipynb`).
The linear median model has no hyperparameters (no penalty) -> only its `seed` can be varied.

In [ ]:
%load_ext autoreload
# --full: reload whole modules. The default only re-runs the changed lines of a module, so a setting computed from an
# edited one (e.g. TASK1_BLENDED_MODELS from TASK1_BLEND_MODEL_FLAGS in lib/config.py) would keep its old value
%autoreload 2 --full

# grid_search.py lives in this folder and imports assignment1.py / lib/ from ../submission
import torch

from grid_search import best_values_of, print_sweep_overview, run_grid_search

print(torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

## LightGBM gap model (CPU)
The linear median model has no hyperparameters (no penalty) -> nothing to sweep.

In [ ]:
if sweep_lightgbm_gap_model:
    # stage 1 - tree size x minimum leaf size (9 runs)
    lightgbm_gap_sweep = run_grid_search("task6", "lightgbm_gap_model", {
        "lightgbm_num_leaves": [15, 31, 63],
        "lightgbm_min_child_samples": [50, 100, 200],
    })
    lightgbm_gap_best_values = best_values_of(lightgbm_gap_sweep)

In [ ]:
if sweep_lightgbm_gap_model:
    # stage 2 - number of trees at learning rate 0.03 (3 runs)
    lightgbm_gap_tree_sweep = run_grid_search("task6", "lightgbm_gap_model", {
        **lightgbm_gap_best_values,
        "lightgbm_n_estimators": [300, 600, 1200],
    })
    lightgbm_gap_best_values |= best_values_of(lightgbm_gap_tree_sweep)

In [ ]:
if sweep_lightgbm_gap_model:
    # stage 3 - noise check: the best settings with other fold assignments (3 runs)
    lightgbm_gap_seed_check = run_grid_search("task6", "lightgbm_gap_model", {
        **lightgbm_gap_best_values,
        "seed": [0, 1, 2],
    })

## Overview

In [ ]:
print_sweep_overview("task6")